# NB04b – Uma semana na Entrega Já
**Métodos e Ferramentas de Data Science – FGV MBA · Aula 4 · Quanto custa um dado ruim**

## A história
Segunda-feira, 8h30. É o seu primeiro dia como analista de dados da **Entrega Já**, uma empresa (fictícia) de comércio eletrônico que vende e entrega produtos em todo o Brasil. A Renata, diretora de dados, te recebe com um café, um arquivo e uma lista:

> *"Seis áreas precisam de um número até sexta-feira. Todas vão decidir com a mesma base de pedidos. Uma coisa eu já te adianto: essa base tem problemas. Não sei quais, nem quantos. Antes de responder qualquer pessoa, olhe a base."*

E ela te passa um segundo arquivo:

> *"Esta é a base da auditoria. Ela conferiu cada pedido do ano, um a um, e só fica pronta depois que as decisões já foram tomadas. Use para ver o que teria acontecido se a empresa decidisse com o dado certo."*

| Parte | Dia | Quem pergunta | A pergunta |
|---|---|---|---|
| 1 | Segunda | Paulo, operações | Que transportadora contratar para 2027? |
| 2 | Terça | Camila, RH | Quantos entregadores contratar no Sudeste? |
| 3 | Quarta | Jorge, financeiro | O time de logística ganhou o bônus? |
| 4 | Quinta de manhã | Beatriz, comercial | Que categoria ganha mais estoque? |
| 5 | Quinta à tarde | Helena, presidente | A meta de satisfação foi batida? |
| 6 | Sexta de manhã | Rafael, marketing | Que prazo prometer no Norte? |
| 7 | Sexta à tarde | Todos | A reunião: o que você vai dizer? |

São a mesma empresa e as mesmas seis decisões do simulador da aula, agora com o pandas.

## Os dados
- **`pedidos`**: a base **como está**, a que a empresa usa para decidir. Cada linha deveria ser um pedido.
- **`auditoria`**: a mesma base **sem nenhum defeito**, a verdade. Os dados foram simulados; por isso conhecemos a verdade. Na vida real, você só teria a primeira.

Os dois arquivos têm 1% dos pedidos do ano (cerca de 500 mil): para chegar ao total do ano, as contas multiplicam por 100.

## Como usar
Execute as células **de cima para baixo**, uma por vez, com **Shift + Enter**.
- **Antes** de cada célula de código, o bloco **"O que acontece na célula abaixo"** explica o código. Leia, execute e **olhe a saída** antes de seguir.
- **Depois** da saída, o bloco **"O que a saída mostra"** traz o resultado e a conclusão. Tente tirar a sua conclusão antes de ler.
- Nas **Questões**, escreva o código na célula vazia logo abaixo. A dica diz qual comando usar.
- Nos blocos **Explore**, mude o código e veja o que acontece. Não tem jeito errado de explorar: se der erro, desfaça a mudança (Ctrl + Z) e execute de novo.

O Python escreve os números com **ponto** decimal (8.70); no texto, usamos vírgula (8,70).

**Aviso:** as regras de decisão da Entrega Já são para fins didáticos apenas, não fórmulas publicadas. Empresa, pessoas, produtos e números são fictícios.

## 1. Ferramentas e as duas bases

**O que acontece na célula abaixo**

- `import pandas as pd` carrega o pandas; `import math` carrega contas simples, como arredondar para cima (`math.ceil`), que vai servir na terça-feira.
- `pd.read_csv(...)` lê cada arquivo e cria uma tabela. O `parse_dates=[...]` pede ao pandas para ler as duas colunas de data já como **datas** (no NB04 fizemos isso depois, com `pd.to_datetime`).
- `.shape` mostra **(linhas, colunas)** de cada tabela.

In [ ]:
import math
import pandas as pd
pd.set_option("display.float_format", "{:.2f}".format)   # números com 2 casas

pedidos = pd.read_csv("https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/entrega_ja_pedidos.csv.gz", parse_dates=["data_compra", "data_entrega"])
auditoria = pd.read_csv("https://raw.githubusercontent.com/carloscarlim/fgv-mfds/main/dados/entrega_ja_auditoria.csv.gz", parse_dates=["data_compra", "data_entrega"])

print("Pedidos (como está):", pedidos.shape)
print("Auditoria (a verdade):", auditoria.shape)

**O que a saída mostra**

As duas tabelas têm as mesmas 11 colunas. Mas a auditoria tem **5.000 linhas** e a base como está, **5.400**. Se a auditoria conferiu os pedidos um a um, de onde vieram 400 linhas a mais? Guarde essa pergunta.

**O que acontece na célula abaixo**

- `pedidos.head()` mostra as 5 primeiras linhas da base como está.

In [ ]:
pedidos.head()

**O que a saída mostra**

Cada linha é um pedido, com estas colunas:

| Coluna | O que guarda |
|---|---|
| `pedido` | número do pedido (P00001, P00002…). Cada pedido tem um número só |
| `data_compra`, `data_entrega` | dia da compra e dia em que o cliente recebeu |
| `transportadora` | quem entregou: A, B ou C |
| `regiao` | região do cliente: SE, S, CO, NE ou N |
| `produto`, `categoria` | o produto comprado e a categoria registrada no pedido |
| `valor` | valor do pedido, em R\$ |
| `prazo_prometido` | dias prometidos na compra, por região: SE 7, S 9, CO 10, NE 12, N 15 |
| `prazo_real` | dias entre a compra e a entrega |
| `nota` | de 1 a 5, da pesquisa de satisfação |

Dois detalhes do processo que vão importar: a **data de entrega** é registrada pela transportadora quando o cliente recebe (é a **confirmação de entrega**), e a **pesquisa de satisfação** só é enviada depois dessa confirmação.

## 2. Primeira olhada: quantos problemas cabem numa base?
Antes de responder qualquer área, você segue o conselho da Renata e olha a base. Na Aula 3, a primeira ideia foi: **olhe a tabela antes da conta**.

> **Conceito · Dado ruim não dá erro**
>
> Uma base pode ter defeitos que não geram nenhuma mensagem de erro: a conta roda e o resultado sai com cara de certo. Os defeitos mais comuns são o **faltante**, a **linha repetida**, o **valor impossível** e a **categoria inconsistente**. Antes de calcular, olhe: quantas linhas, o que falta, o menor e o maior valor, como as categorias estão escritas (McKinney, 2022, cap. 7).

**O que acontece na célula abaixo**

- `pedidos["pedido"].nunique()` conta quantos **números de pedido diferentes** existem. Se cada pedido tem um número só, deveria dar o mesmo que o número de linhas.

In [ ]:
pedidos["pedido"].nunique()

**O que a saída mostra**

**5.000 números diferentes em 5.400 linhas.** Há 400 linhas cujo número de pedido já apareceu antes. **Pista 1.**

**O que acontece na célula abaixo**

- `pedidos.isna().sum()` conta os valores vazios de cada coluna, como no NB04.

In [ ]:
pedidos.isna().sum()

**O que a saída mostra**

**808 vazios** em `data_entrega`, `prazo_real` e `nota`, sempre juntos: são os pedidos sem confirmação de entrega. Sem a data, não há prazo; sem a confirmação, a pesquisa não sai. **Pista 2.**

**O que acontece na célula abaixo**

- `pedidos["prazo_real"].describe()` resume o prazo: quantidade, média, desvio, menor valor, quartis e maior valor.

In [ ]:
pedidos["prazo_real"].describe()

**O que a saída mostra**

O **menor prazo é −15 dias** e o **maior, 3.668 dias** (dez anos!). Nenhum dos dois é possível para uma entrega. E veja a média, 67,5 dias, contra uma mediana de 7: poucos valores absurdos puxam a média para cima. **Pista 3.**

Três pistas em três linhas de código. Haverá uma quarta, que nenhuma destas contas mostra: nem todo defeito aparece na primeira olhada. Cada pista vai reaparecer numa parte da semana.

---
## Parte 1 · Segunda-feira: a transportadora que (quase) nunca atrasa
10h. O Paulo, de operações, aparece na sua mesa:

> *"Vou concentrar todas as entregas de 2027 numa transportadora só: a que atrasa menos. Cada atraso me custa um cupom de R\$ 20 para o cliente. Preciso da taxa de atraso de cada uma até o fim do dia."*

**A regra da empresa:** taxa de atraso = pedidos que atrasaram ÷ pedidos **com confirmação de entrega**. O pedido sem confirmação fica de fora, porque não tem prazo real.

**O que acontece na célula abaixo**

- A primeira linha cria a coluna `sem_confirmacao`: `True` quando a data de entrega está vazia.
- `groupby("transportadora")["sem_confirmacao"].mean()` calcula, para cada transportadora, a **proporção** de pedidos sem confirmação (a média de `True`/`False` é a proporção de `True`, como no NB04). O `* 100` passa para %.

In [ ]:
pedidos["sem_confirmacao"] = pedidos["data_entrega"].isna()
pedidos.groupby("transportadora")["sem_confirmacao"].mean() * 100

**O que a saída mostra**

A **C deixa 27,7% dos pedidos sem confirmação**; a A, 9,4%; a B, 12,1%. Uma em cada quatro entregas da C não tem registro. Isso deveria acender um alerta antes de qualquer conta.

**O que acontece na célula abaixo**

Agora, a conta da empresa:
- `atrasou` é `True` quando o prazo real passa do prometido. Com o prazo vazio, a comparação dá `False`.
- `confirmados` fica só com os pedidos que têm data de entrega (`notna()` é o contrário de `isna()`).
- A taxa de atraso é a proporção de `True` em `atrasou`, por transportadora, em %.

In [ ]:
pedidos["atrasou"] = pedidos["prazo_real"] > pedidos["prazo_prometido"]
confirmados = pedidos[pedidos["data_entrega"].notna()]
confirmados.groupby("transportadora")["atrasou"].mean() * 100

**O que a saída mostra**

A **C tem a menor taxa: 4,7%**, contra 8,2% da A e 10,8% da B. Pela regra, o Paulo contrataria a **C**.

Mas é a mesma C que deixa quase 28% dos pedidos sem registro. Coincidência?

**O que acontece na célula abaixo**

- A mesma conta, agora com a `auditoria`, em que todos os pedidos têm confirmação.

In [ ]:
auditoria["atrasou"] = auditoria["prazo_real"] > auditoria["prazo_prometido"]
auditoria.groupby("transportadora")["atrasou"].mean() * 100

**O que a saída mostra**

Na verdade, a **C é a pior: 23,5% de atraso**. A A atrasa 8,7% e a B, 13,6%. A decisão certa era a **A**. O dado ruim não errou um pouco: **inverteu** o resultado, e a pior transportadora virou a melhor.

**Questão 1.** Entre os pedidos da **C** que ficaram sem confirmação, que parte, **na auditoria**, atrasou?
Dica: guarde os números desses pedidos e procure-os na auditoria com `isin` ("está na lista?"):
```python
ids = pedidos[pedidos["sem_confirmacao"] & (pedidos["transportadora"] == "C")]["pedido"]
auditoria[auditoria["pedido"].isin(ids)]["atrasou"].mean()
```

In [ ]:
# Questão 1: escreva aqui


> **Conceito · Por que o dado falta?**
>
> Antes de tratar um faltante, pergunte **por que** ele falta. Ele pode faltar **ao acaso** (um lote de registros se perdeu), **por algo que você vê na base** (uma região responde menos) ou **pelo próprio valor que falta** (a entrega atrasada não é registrada). Rubin (1976) estudou quando é possível ignorar o motivo da falta: quando ela depende do próprio valor que falta, ignorar os faltantes **distorce o resultado**, como aconteceu com a C.

**O que acontece na célula abaixo**

Um **tratamento** possível, o do pior caso: contar todo pedido sem confirmação como atraso.
- `|` quer dizer **OU**: a nova coluna é `True` se o pedido atrasou **ou** ficou sem confirmação.
- Agora a conta usa **todos** os pedidos, sem tirar ninguém.

In [ ]:
pedidos["atrasou_ou_sem_confirmacao"] = pedidos["atrasou"] | pedidos["sem_confirmacao"]
pedidos.groupby("transportadora")["atrasou_ou_sem_confirmacao"].mean() * 100

**O que a saída mostra**

A **A volta a vencer** (16,9%, contra 21,5% da B e 31,1% da C): o tratamento **acertou a decisão**. Mas **errou o número**: a A atrasa 8,7%, e não 16,9%, porque nem todo pedido sem confirmação atrasou. É uma hipótese de pior caso: serve para testar se a decisão muda, não para informar a taxa.

> **Explore.** Troque `"transportadora"` por `"regiao"` na primeira célula da Parte 1: alguma região tem mais pedidos sem confirmação? E troque `.mean()` por `.sum()`: o que passa a aparecer? Depois, olhe 10 pedidos da C com `pedidos[pedidos["transportadora"] == "C"].head(10)`.

**Para levar à reunião.** Decidir com a base como está = contratar a C. Na auditoria, a C atrasa 14,8 pontos percentuais a mais que a A: em 500 mil pedidos, cerca de **74 mil atrasos a mais por ano** e, com o cupom de R\$ 20, cerca de **R\$ 1,5 milhão por ano**. E uma pergunta para a C antes de qualquer contrato: *por que um em cada quatro pedidos de vocês fica sem registro?*

---
## Parte 2 · Terça-feira: 400 pedidos que não existem
9h. A Camila, do RH, precisa contratar entregadores próprios para o Sudeste:

> *"Cada entregador dá conta de 40 pedidos por dia e custa R\$ 4 mil por mês. Quantos eu contrato? E, por favor, me diga o custo do ano."*

**A regra da empresa:** pedidos por dia = linhas do Sudeste na base × 100 ÷ 365. Entregadores = pedidos por dia ÷ 40, arredondado para cima.

Você lembra da **pista 1**: 400 linhas a mais. Desde março, parte dos pedidos chega à base por uma integração com um marketplace.

**O que acontece na célula abaixo**

- `sudeste` fica só com as linhas da região SE.
- `len(sudeste)` conta as **linhas**; `.nunique()` conta os **números de pedido diferentes**.

In [ ]:
sudeste = pedidos[pedidos["regiao"] == "SE"]
print("Linhas do Sudeste:  ", len(sudeste))
print("Pedidos diferentes: ", sudeste["pedido"].nunique())

**O que a saída mostra**

**2.628 linhas, mas só 2.228 pedidos diferentes.** As 400 linhas a mais estão todas no Sudeste.

**O que acontece na célula abaixo**

- `duplicated(keep=False)` marca com `True` **todas** as linhas cujo número de pedido aparece mais de uma vez (sem o `keep=False`, só a segunda vez seria marcada).
- A lista entre colchetes escolhe 5 colunas para caber na tela; `.head(6)` mostra as 6 primeiras linhas.

In [ ]:
repetidos = sudeste[sudeste["pedido"].duplicated(keep=False)]
repetidos[["pedido", "data_compra", "produto", "valor", "transportadora"]].head(6)

**O que a saída mostra**

As linhas vêm **em pares idênticos**: mesmo número, mesma data, mesmo produto, mesmo valor. É o mesmo pedido gravado duas vezes pela integração. Não são 400 clientes a mais.

**O que acontece na célula abaixo**

A conta da Camila, passo a passo, primeiro com a base como está:
- `math.ceil` arredonda **para cima**: se a conta der 17,1 entregadores, a empresa contrata 18 (não dá para contratar um pedaço de pessoa).

In [ ]:
linhas = len(sudeste)
pedidos_por_dia = linhas * 100 / 365
entregadores = math.ceil(pedidos_por_dia / 40)
print("Pedidos por dia:", round(pedidos_por_dia, 1))
print("Entregadores:", entregadores)
print("Custo por ano: R$", entregadores * 4000 * 12)

**O que a saída mostra**

720 pedidos por dia, **18 entregadores**, **R\$ 864 mil por ano**.

> **Conceito · A chave e a linha repetida**
>
> A **chave** é a coluna que identifica a unidade da tabela: aqui, o número do pedido. Uma chave repetida infla contagens e somas. Mas cuidado: **linha repetida nem sempre é erro**. Na base do NB04, cada linha era um item, e um pedido com 3 itens aparecia 3 vezes. Antes de remover, pergunte: o que cada linha representa? No pandas: `duplicated()` encontra e `drop_duplicates(subset=...)` remove (McKinney, 2022, seção 7.2).

**O que acontece na célula abaixo**

**O tratamento:** `drop_duplicates(subset="pedido")` deixa uma linha por número de pedido. Depois, a mesma conta, e a da auditoria para comparar.

In [ ]:
sem_repeticao = pedidos.drop_duplicates(subset="pedido")
linhas_tratadas = len(sem_repeticao[sem_repeticao["regiao"] == "SE"])
linhas_auditoria = len(auditoria[auditoria["regiao"] == "SE"])
print("Tratado:  ", linhas_tratadas, "linhas →", math.ceil(linhas_tratadas * 100 / 365 / 40), "entregadores")
print("Auditoria:", linhas_auditoria, "linhas →", math.ceil(linhas_auditoria * 100 / 365 / 40), "entregadores")

**O que a saída mostra**

Sem as repetições, **16 entregadores**, exatamente como na auditoria. Aqui o tratamento acerta a decisão **e** o número, porque a repetição é fácil de reconhecer: o número do pedido.

**Questão 2.** Quanto a empresa gastaria **a mais por ano** se contratasse com a base como está?
Dica: (18 − 16) entregadores × R\$ 4.000 × 12 meses. Escreva a conta em Python.

In [ ]:
# Questão 2: escreva aqui


> **Explore.** Calcule `pedidos["pedido"].duplicated().sum()` e depois `pedidos.duplicated().sum()` (sem escolher a coluna): o pandas passa a comparar a **linha inteira**. O número de repetidas muda? Depois, compare `pedidos["regiao"].value_counts()` com `sem_repeticao["regiao"].value_counts()`: que região muda?

**Para levar à reunião.** 2 entregadores a mais, R\$ 96 mil por ano. E uma regra de qualidade para a TI: *o número do pedido é chave única; a base não aceita o mesmo número duas vezes.*

---
## Parte 3 · Quarta-feira: o bônus perdido em 2036
11h. O Jorge, do financeiro, precisa fechar a folha:

> *"O time de logística ganha R\$ 300 mil de bônus se o prazo médio de entrega do ano ficar em até 10 dias. Ganhou ou não ganhou?"*

**A regra da empresa:** prazo médio = média do prazo real de todos os pedidos com data de entrega. Paga o bônus se for de até 10 dias.

As datas de compra e de entrega são digitadas num formulário pelos parceiros de entrega. E você lembra da **pista 3**: prazos de −15 e de 3.668 dias.

**O que acontece na célula abaixo**

- `sort_values("prazo_real")` ordena a tabela pelo prazo, do menor para o maior.
- A lista entre colchetes escolhe só as 4 colunas que interessam; `.head()` mostra as 5 primeiras linhas (os menores prazos).

In [ ]:
colunas = ["pedido", "data_compra", "data_entrega", "prazo_real"]
pedidos.sort_values("prazo_real")[colunas].head()

**O que a saída mostra**

Nos menores prazos, a **data de entrega vem antes da compra**: as duas datas foram digitadas uma no lugar da outra. Prazo negativo não existe.

**O que acontece na célula abaixo**

- O mesmo, com `ascending=False`: a ordem fica do **maior** para o menor, e o `.head()` mostra os maiores prazos. (Os pedidos sem prazo ficam sempre no fim da ordem.)

In [ ]:
pedidos.sort_values("prazo_real", ascending=False)[colunas].head()

**O que a saída mostra**

Nos maiores, a entrega está em **2036**: o ano foi digitado errado. Cada um desses pedidos soma cerca de 3.650 dias à conta.

**O que acontece na célula abaixo**

- A conta do Jorge: a média do prazo, como está e na auditoria. O `round(..., 1)` mostra com 1 casa decimal.

In [ ]:
print("Prazo médio, como está:", round(pedidos["prazo_real"].mean(), 1), "dias")
print("Prazo médio, auditoria:", round(auditoria["prazo_real"].mean(), 1), "dias")

**O que a saída mostra**

Como está, **67,5 dias**: o time não ganha o bônus. Na auditoria, **8,5 dias**: o time **cumpriu a meta**. Pouco mais de cem datas digitadas errado tiram R\$ 300 mil de quem fez o trabalho.

> **Conceito · Valor impossível e regra de validação**
>
> Um **valor impossível** está preenchido, mas fora do que é possível: entrega antes da compra, entrega dez anos depois. É diferente do *outlier*, o valor extremo que é raro, mas pode ser verdadeiro (NB04, seção 13). Uma **regra de validação** define o intervalo possível de uma coluna e encontra esses valores antes da conta. O `describe()` (menor e maior valor) é o primeiro detector.

**O que acontece na célula abaixo**

**Tratamento 1, a regra de validação:** ficar só com os prazos entre 0 e 120 dias.
- `&` quer dizer **E**: as duas condições precisam ser verdadeiras. Cada condição vai entre parênteses.

In [ ]:
validos = pedidos[(pedidos["prazo_real"] >= 0) & (pedidos["prazo_real"] <= 120)]
print("Pedidos válidos:", len(validos))
print("Prazo médio dos válidos:", round(validos["prazo_real"].mean(), 1), "dias")

**O que a saída mostra**

Com a regra de validação, **7,9 dias**: o time ganha o bônus, como na auditoria. Mas repare: 7,9 é **menor** que os 8,5 da verdade. Por quê? Lembre da segunda-feira: os pedidos **sem confirmação**, quase sempre atrasados, também estão fora desta conta. **Os defeitos se somam**: tratar um não resolve o outro.

**O que acontece na célula abaixo**

**Tratamento 2, sem apagar nada:** trocar a média pela **mediana**, o valor do meio, que resiste a extremos.

In [ ]:
print("Mediana, como está:", pedidos["prazo_real"].median(), "dias")
print("Mediana, auditoria:", auditoria["prazo_real"].median(), "dias")

**O que a saída mostra**

**7 dias** nas duas bases. A mediana nem percebe as datas absurdas: elas estão na ponta, e a mediana olha o meio. Ela acerta a decisão **sem limpar** a base, mas as datas erradas continuam lá.

**Questão 3.** Quantos prazos da base são impossíveis, isto é, **negativos ou acima de 120 dias**?
Dica: `|` quer dizer **OU**. Compare `(pedidos["prazo_real"] < 0) | (pedidos["prazo_real"] > 120)` e some os `True` com `.sum()`.

In [ ]:
# Questão 3: escreva aqui


> **Explore.** Na célula da regra de validação, troque `120` por `60` e depois por `30`. Quantos pedidos saem? O prazo médio muda muito? (Dica: compare com a Questão 3; as datas erradas estão muito longe de qualquer limite razoável.) Depois, calcule o percentil 90 com `pedidos["prazo_real"].quantile(0.9)`: ele resiste às datas erradas como a mediana?

**Para levar à reunião.** Com a base como está, o time que cumpriu a meta **perde R\$ 300 mil**. A regra de qualidade é simples: *o formulário não aceita entrega antes da compra nem prazo acima de 120 dias.* E uma discussão para o Jorge: a meta deveria ser pela média (que sente cada atraso longo) ou pela mediana (que resiste a erros, mas ignora a cauda)?

---
## Parte 4 · Quinta de manhã: o fone de ouvido na seção de Casa
9h30. A Beatriz, do comercial, vai ampliar o estoque de uma categoria:

> *"Quero apostar na categoria que mais faturou no ano. Me diga qual é."*

**A regra da empresa:** faturamento de cada categoria = soma do valor dos pedidos × 100, agrupada pela categoria escrita no pedido.

Desde março, os vendedores do marketplace cadastram os próprios produtos e escolhem a categoria numa lista.

**O que acontece na célula abaixo**

- `groupby("categoria")["valor"].sum()` soma o valor por categoria; `* 100` leva ao ano; `/ 1_000_000` mostra em **milhões de reais** (o `_` só separa os zeros para facilitar a leitura).
- `sort_values(ascending=False)` ordena do maior para o menor.

In [ ]:
(pedidos.groupby("categoria")["valor"].sum() * 100 / 1_000_000).sort_values(ascending=False)

**O que a saída mostra**

**Casa lidera, com R\$ 25,4 milhões**; Eletrônicos vem atrás, com R\$ 17,8 milhões. Pela regra, a Beatriz ampliaria **Casa**.

**O que acontece na célula abaixo**

- A mesma conta com a auditoria.

In [ ]:
(auditoria.groupby("categoria")["valor"].sum() * 100 / 1_000_000).sort_values(ascending=False)

**O que a saída mostra**

Na verdade, **Eletrônicos lidera**, com R\$ 21,9 milhões, e Casa fatura R\$ 18,0 milhões. Esta é a **pista 4**, a que a primeira olhada não mostrou: nenhum valor está vazio, nenhum número é absurdo. O defeito está no **rótulo**.

**O que acontece na célula abaixo**

Investigue: que **produtos** aparecem nos pedidos registrados como Casa?
- O filtro fica só com os pedidos de Casa; `value_counts()` conta cada produto.

In [ ]:
pedidos[pedidos["categoria"] == "Casa"]["produto"].value_counts()

**O que a saída mostra**

Entre toalhas, panelas e tapetes, aparecem **fone Bluetooth, smartwatch, caixa de som, carregador e teclado**: eletrônicos registrados como Casa. Um vendedor do marketplace escolheu a categoria errada, e o faturamento dele foi parar no grupo errado.

> **Conceito · Categoria inconsistente e tabela de-para**
>
> A categoria fica **inconsistente** quando a mesma coisa é escrita de jeitos diferentes (“casa”, “CASA”) ou quando uma coisa vai para a categoria errada (um fone em “Casa”). Na hora de agrupar, a soma vai para o grupo errado. A correção usa uma **tabela de referência**, a de-para: cada produto com a sua categoria oficial. No pandas, o `map` aplica a tabela, linha a linha (McKinney, 2022, seção 7.2).

**O que acontece na célula abaixo**

**O tratamento:** o cadastro oficial da empresa, num **dicionário** (como na NB01a, seção 5): cada produto aponta para a sua categoria.
- `pedidos["produto"].map(cadastro)` troca cada produto pela sua categoria oficial e cria a coluna `categoria_certa`.
- Depois, a mesma conta do faturamento, agora com a categoria certa.

In [ ]:
cadastro = {
    "Jogo de toalhas": "Casa", "Panela antiaderente": "Casa", "Luminária de mesa": "Casa",
    "Tapete de sala": "Casa", "Organizador de gavetas": "Casa",
    "Fone Bluetooth": "Eletrônicos", "Caixa de som portátil": "Eletrônicos", "Smartwatch": "Eletrônicos",
    "Carregador portátil": "Eletrônicos", "Teclado sem fio": "Eletrônicos",
    "Perfume": "Beleza", "Hidratante": "Beleza", "Protetor solar": "Beleza", "Kit de maquiagem": "Beleza", "Shampoo": "Beleza",
    "Tênis de corrida": "Esporte", "Garrafa térmica": "Esporte", "Bola de futebol": "Esporte",
    "Tapete de ioga": "Esporte", "Corda de pular": "Esporte",
    "Quebra-cabeça": "Brinquedos", "Boneca": "Brinquedos", "Carrinho de controle": "Brinquedos",
    "Jogo de tabuleiro": "Brinquedos", "Blocos de montar": "Brinquedos",
}
pedidos["categoria_certa"] = pedidos["produto"].map(cadastro)
(pedidos.groupby("categoria_certa")["valor"].sum() * 100 / 1_000_000).sort_values(ascending=False)

**O que a saída mostra**

**Eletrônicos volta a liderar** (R\$ 23,8 milhões, contra R\$ 19,4 milhões de Casa): a decisão está certa. Mas os dois números estão **acima** da auditoria (21,9 e 18,0). Algum defeito da semana ainda está nesta conta. Qual?

**Questão 4.** Refaça a conta com a categoria certa, mas **sem as linhas repetidas** da terça-feira. Os números batem com a auditoria?
Dica: comece com `pedidos.drop_duplicates(subset="pedido")` e continue com o mesmo `groupby` da célula anterior.

In [ ]:
# Questão 4: escreva aqui


> **Explore.** A tabela `pd.crosstab(pedidos["produto"], pedidos["categoria"])` cruza produto e categoria: cada célula conta quantos pedidos têm aquele produto naquela categoria. Execute numa célula nova: os erros de cadastro aparecem de uma vez. Depois, conte quantos pedidos mudaram de categoria com `(pedidos["categoria"] != pedidos["categoria_certa"]).sum()`.

**Para levar à reunião.** Com a base como está, a Beatriz amplia **Casa**, que na verdade fatura cerca de **R\$ 3,9 milhões por ano a menos** que Eletrônicos. A regra de qualidade: *a categoria do pedido vem do cadastro do produto, mantido pela empresa, e não da escolha do vendedor.*

---
## Parte 5 · Quinta à tarde: a nota de quem não respondeu
16h. A Helena, presidente da empresa, vai ao conselho na semana que vem:

> *"O conselho cobra uma nota de satisfação de pelo menos 4,2. Batemos a meta?"*

**A regra da empresa:** nota reportada = média das notas recebidas na pesquisa. A pesquisa sai por e-mail **depois da confirmação de entrega**.

Você lembra da **pista 2** e da segunda-feira: 808 pedidos sem confirmação, e portanto sem nota.

**O que acontece na célula abaixo**

- `pedidos["nota"].mean()` é a nota reportada (a média ignora os vazios sozinha).
- `notna().mean()` é a **taxa de resposta**: a parte dos pedidos que tem nota.
- A última linha faz a conta na auditoria, em que todos os clientes responderam.

In [ ]:
print("Nota reportada:", round(pedidos["nota"].mean(), 2))
print("Taxa de resposta:", round(pedidos["nota"].notna().mean(), 2))
print("Nota na auditoria:", round(auditoria["nota"].mean(), 2))

**O que a saída mostra**

Como está, **4,34: meta batida**, com 85% de resposta. Na auditoria, **4,16: meta não batida**. A Helena levaria ao conselho uma meta que não foi cumprida.

**O que acontece na célula abaixo**

Quem respondeu e quem não respondeu? Comece pela nota de quem atrasou e de quem não atrasou:
- `groupby("atrasou")["nota"].mean()` dá a nota média de cada grupo (`False` = não atrasou; `True` = atrasou). Os pedidos sem confirmação ficam em `False`, mas não têm nota, então não entram na média.

In [ ]:
pedidos.groupby("atrasou")["nota"].mean()

**O que a saída mostra**

Quem recebe no prazo dá **4,50**; quem recebe com atraso, **2,51**. E quem sumiu da pesquisa? Na segunda-feira você viu que os pedidos sem confirmação são, na maioria, **atrasados**. Os clientes mais insatisfeitos são justamente os que não receberam a pesquisa.

> **Conceito · Quem não responde muda a média**
>
> Quando a falta de resposta depende da própria resposta (o insatisfeito não responde), a média de quem respondeu **não representa todos** (Rubin, 1976). Preencher o vazio com a **média de quem respondeu** parece um conserto, mas **não muda a média**: só esconde o problema. Uma alternativa honesta é testar **cenários**: e se quem não respondeu desse uma nota parecida com a de quem atrasou? Se a decisão muda com uma hipótese razoável, o número não é confiável.

**O que acontece na célula abaixo**

Dois tratamentos com `fillna` (preencher os vazios), lado a lado:
- com a **média de quem respondeu**;
- com a **média de quem atrasou** (`pedidos[pedidos["atrasou"]]` fica só com os atrasados), uma hipótese: quem ficou sem confirmação se parece com quem atrasou.

In [ ]:
media_respondeu = pedidos["nota"].mean()
media_atrasou = pedidos[pedidos["atrasou"]]["nota"].mean()
print("Preenchendo com a média de quem respondeu:", round(pedidos["nota"].fillna(media_respondeu).mean(), 2))
print("Preenchendo com a média de quem atrasou:  ", round(pedidos["nota"].fillna(media_atrasou).mean(), 2))

**O que a saída mostra**

Com a média de quem respondeu, a nota continua **4,34**: preencher pela média não muda a média. Com a média de quem atrasou, a nota cai para **4,06**: a decisão fica certa (meta não batida), mas o número fica **abaixo** da verdade (4,16), porque nem todo pedido sem confirmação atrasou. O tratamento é uma hipótese, e precisa ser dito como hipótese.

**Questão 5.** Teste dois cenários: e se todos que não responderam dessem nota **1**? E nota **3**?
Dica: `pedidos["nota"].fillna(1).mean()` e o mesmo com 3. Em qual cenário a meta ainda seria batida?

In [ ]:
# Questão 5: escreva aqui


> **Explore.** Troque `"atrasou"` por `"transportadora"` no `groupby` da nota. Que transportadora tem a melhor nota? Lembre da segunda-feira: faz sentido? Depois, preencha os vazios com a **mediana** (`pedidos["nota"].median()`): a nota sobe ou desce? Por quê?

**Para levar à reunião.** Com a base como está, a Helena diria ao conselho que a meta foi batida (4,34), quando a nota de todos é 4,16. Uma forma honesta de reportar: *a nota de quem respondeu, a taxa de resposta (85%) e a nota separada entre pedidos no prazo e atrasados.*

---
## Parte 6 · Sexta de manhã: a promessa de 690 dias
9h. O Rafael, do marketing, vai mostrar na página de compra o prazo de entrega para a região Norte:

> *"Usamos a mesma regra há anos: a média do prazo real mais um desvio padrão, arredondado para cima. Qual prazo eu coloco no site?"*

**A regra da empresa:** prazo prometido = média + 1 desvio padrão do prazo real dos pedidos do Norte. Prometer demais afasta o cliente na compra; prometer de menos gera atraso e nota baixa depois.

**O que acontece na célula abaixo**

- `norte` guarda só os prazos dos pedidos da região N; `.describe()` resume.

In [ ]:
norte = pedidos[pedidos["regiao"] == "N"]["prazo_real"]
norte.describe()

**O que a saída mostra**

Média de **107,1 dias** e desvio padrão de **582,2**, mas mediana de **12**. As datas digitadas errado da quarta-feira também estão aqui (o maior prazo é 3.668 dias).

**O que acontece na célula abaixo**

- A regra da empresa, como está e na auditoria. `math.ceil` arredonda para cima.

In [ ]:
norte_auditoria = auditoria[auditoria["regiao"] == "N"]["prazo_real"]
print("Como está:", math.ceil(norte.mean() + norte.std()), "dias")
print("Auditoria:", math.ceil(norte_auditoria.mean() + norte_auditoria.std()), "dias")

**O que a saída mostra**

Como está, o Rafael prometeria **690 dias**, quase dois anos. Com o dado certo, **17 dias**. A média e o desvio padrão **explodem** com poucas datas absurdas.

> **Conceito · Medidas robustas**
>
> A **média** e o **desvio padrão** sofrem com valores extremos: um único erro pode mudá-los muito. A **mediana** (o valor do meio) e o **desvio absoluto mediano** (MAD, a mediana das distâncias de cada valor até a mediana) **resistem**. Multiplicado por 1,4826, o MAD fica na mesma escala do desvio padrão quando os dados se parecem com a curva normal (Rousseeuw e Croux, 1993). Robusta não quer dizer corrigida: o erro continua na base.

**O que acontece na célula abaixo**

**A regra robusta:** mediana + 1,4826 × MAD.
- `(norte - mediana).abs()` calcula a distância de cada prazo até a mediana (`.abs()` tira o sinal); `.median()` pega a mediana dessas distâncias: é o MAD.

In [ ]:
mediana = norte.median()
mad = (norte - mediana).abs().median()
print("Mediana:", mediana, "| MAD:", mad)
print("Prazo pela regra robusta:", math.ceil(mediana + 1.4826 * mad), "dias")

**O que a saída mostra**

Mediana de 12, MAD de 2: a regra robusta promete **15 dias**, mesmo com as datas absurdas na base. Na auditoria, ela daria o mesmo.

**O que acontece na célula abaixo**

Que parte dos pedidos do Norte, **na verdade**, chegaria dentro de cada prazo prometido?
- `norte_auditoria <= 17` dá `True` para os pedidos entregues em até 17 dias; `.mean()` é a proporção.

In [ ]:
print("Em até 690 dias:", round((norte_auditoria <= 690).mean(), 2))
print("Em até 17 dias: ", round((norte_auditoria <= 17).mean(), 2))
print("Em até 15 dias: ", round((norte_auditoria <= 15).mean(), 2))

**O que a saída mostra**

Com 690 dias, todos chegam no prazo, mas nenhum cliente compra com essa promessa. Com 17 dias, **89%** chegam no prazo; com 15, **87%**. A diferença entre as duas últimas é pequena; a da primeira é absurda.

**Questão 6.** Calcule o **percentil 90** do prazo do Norte, como está e na auditoria: o prazo que 90% dos pedidos cumprem.
Dica: `.quantile(0.9)`, como no NB04 (seção 11), em `norte` e em `norte_auditoria`.

In [ ]:
# Questão 6: escreva aqui


> **Explore.** Troque `0.9` por `0.8` e por `0.95` na Questão 6: como o prazo muda? Que percentual de pedidos no prazo você prometeria ao cliente? Depois, troque `1.4826 * mad` por `2 * 1.4826 * mad`: o que acontece com o prazo?

**Para levar à reunião.** Com a base como está, o site prometeria **690 dias**. Uma regra melhor para o Rafael, e que a operação entende: *prometer o prazo que 90% dos pedidos cumprem* (o P90), calculado com a base tratada.

---
## Parte 7 · Sexta à tarde: a reunião
16h. As seis áreas e a Renata estão na sala. Você tem cinco minutos. Complete a tabela (dê dois cliques nesta célula para editar):

| Parte | Decisão com a base como está | Decisão com o dado certo | O defeito | Consequência |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |

E responda, em uma frase cada:
1. Alguma conta deu erro durante a semana? Então, por que seis decisões erraram?
2. Qual defeito mudou uma decisão com a **menor** quantidade de dado ruim?
3. Em que partes o defeito **inverteu** o resultado? O que essas partes têm em comum?
4. Em que partes o tratamento **acertou a decisão, mas errou o número**?
5. O que a medida robusta resolveu, e o que ela não resolveu?
6. Que regra de qualidade você pediria à TI para cada defeito?

**Referências:** McKINNEY, W. *Python for Data Analysis*. 3. ed. O’Reilly, 2022, cap. 7 (Data Cleaning and Preparation). RUBIN, D. B. Inference and missing data. *Biometrika*, 63(3), 1976. ROUSSEEUW, P. J.; CROUX, C. Alternatives to the median absolute deviation. *Journal of the American Statistical Association*, 88(424), 1993.

## Explore mais
Se sobrou tempo, tente estas perguntas. Cada uma pede só uma ou duas linhas, com os comandos desta semana.
- A taxa de atraso da segunda-feira muda se você usar só os pedidos `validos` da quarta-feira (sem as datas impossíveis)?
- As 400 linhas repetidas mexem na nota da quinta-feira? Calcule a nota com `sem_repeticao`.
- Quantos pedidos têm **mais de um** defeito? (Dica: um pedido sem confirmação pode estar repetido.)
- Qual é a transportadora com mais datas impossíveis? Use o filtro da Questão 3 e `value_counts()`.

## Erros comuns e como resolver

| Mensagem | O que aconteceu | O que fazer |
|---|---|---|
| `NameError: name 'pedidos' is not defined` | A primeira célula não foi executada | Execute as células desde o início (*Ambiente de execução → Executar anteriores*) |
| `NameError` com `sem_repeticao`, `validos`, `norte` ou `cadastro` | A célula que cria esse nome não foi executada | Execute as células anteriores da parte |
| `KeyError: 'atrasou'` ou `'sem_confirmacao'` | A Parte 1 não foi executada | Essas colunas nascem na Parte 1: execute-a antes |
| `SyntaxError` | Faltou parêntese, aspas ou dois-pontos, ou um decimal com vírgula (`0,9`) | Confira a linha indicada; use ponto: `0.9` |
| `TypeError: unsupported operand type(s) for &` | Faltaram os parênteses em volta de cada condição | Escreva `(condição 1) & (condição 2)` |
| `HTTPError` ou `URLError` na primeira célula | Sem internet ou rede bloqueada | Use o Plano B abaixo |

## Plano B: se o download falhar
1. Baixe os arquivos `entrega_ja_pedidos.csv.gz` e `entrega_ja_auditoria.csv.gz` da pasta `dados/` do repositório da disciplina (ou pelo link do ECLASS).
2. No Colab, clique no ícone de **pasta** na barra lateral esquerda e arraste os dois arquivos para lá.
3. Na primeira célula, apague o endereço que vem antes do nome de cada arquivo, deixando só `"entrega_ja_pedidos.csv.gz"` e `"entrega_ja_auditoria.csv.gz"`, e execute de novo.

### Autor e licença

**Autor:** Prof. Carlos Carlim · [LinkedIn](https://www.linkedin.com/in/carloscarlim-77144120) · [GitHub](https://github.com/carloscarlim)

**Licença MIT.** Copyright (c) 2026 Carlos Carlim. Os direitos autorais deste material pertencem exclusivamente ao autor.

É permitido usar, copiar, modificar, mesclar, publicar, distribuir, sublicenciar e vender cópias deste material, desde que este aviso de copyright e o texto da licença sejam mantidos em todas as cópias ou partes substanciais. O material é fornecido "no estado em que se encontra", sem garantia de qualquer tipo. Texto completo da licença: [LICENSE](https://github.com/carloscarlim/fgv-mfds/blob/main/LICENSE).

Os dados usados nos notebooks têm licenças próprias, indicadas em [`dados/README.md`](https://github.com/carloscarlim/fgv-mfds/blob/main/dados/README.md).